## Imports

In [1]:
import json
import os
import re
from datetime import datetime

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB

## Config And Parameters

All parameters are in this code cell below, categorized. Text cleaning uses regex to preprocess the input and replace latex or URLs. Found to be useful and mostly left on, but gradually it seemed to decay.

Validation split controls the how much of the training set is used for testing, and contains the rng seed.

TF IDF Vectorizer controls all of its respective parameters, such as the number of features (words), min_df controls how many times a word needs to appear to be counted as a feature, whereas max_df controls what percentage of rows the feature must appear before it no longer is counted as a feature (for being non-distinguishing and not very useful due to commonality).

NGRAM Range is controls how many adjacent terms/features/words can be grouped as a single feature. These sequences are order dependent. Currently using Bigrams. Occassional arbitrary/stochastic testing of using trigrams yielded negative results, including trying with higher feature counts.

Sublinear TF. Apply log scaling to counts. Found stochastically after finding a local optimal without using it to bring improvements.

Alpha: Laplace scaling as explored in the homework. Originally only tested 0.1, 0.3, 0.5, 1.0. But after seeing some good results on some combos with 0.1 decided I should decrease the lower bound and explore lower values. Sweeps of these found 0.001 to be good.



In [75]:
CONFIG = {
    # I/O
    'DATA_DIR': './',
    'OUTPUT_FILE': 'MultinomialNB_Prediction.csv',
    'PARAMETERS_FILE': 'MultinomialNB_Parameters.json',
    'LOG_FILE': 'MultinomialNB_RunLog.txt',

    # Pipeline switches
    'CHECK_TRAINING_SCORE': False,
    'EXPORT_SUBMISSION': True,
    'STAMP_PREDICTION_FILE': True,
    'LOG_RUN': True,
    'SKIP_IF_ALREADY_LOGGED': True,

    # Text cleaning
    'USE_SAFE_TEXT_CLEANING': True,
    'STRIP_LATEX_MATH': True,
    'STRIP_URLS': True,
    'NORMALIZE_WHITESPACE': True,

    # Validation split
    'VAL_FRACTION': 0.2,
    'RANDOM_STATE': 42,

    # TfidfVectorizer
    'MAX_FEATURES': 750000,
    'MIN_DF': 2,
    'MAX_DF': 0.85,
    'STOP_WORDS': 'english',
    'NGRAM_RANGE': (1, 2),
    'SUBLINEAR_TF': False,

    # Naive Bayes model selection
    # 'multinomial' = MultinomialNB (uses ALPHA, FIT_PRIOR; ignores NORM)
    # 'complement'  = ComplementNB  (uses ALPHA, FIT_PRIOR, NORM)
    'MODEL': 'multinomial',
    'NORM': False,
    'ALPHA': 0.003,
    'FIT_PRIOR': False,
}

CONFIG

{'DATA_DIR': './',
 'OUTPUT_FILE': 'MultinomialNB_Prediction.csv',
 'PARAMETERS_FILE': 'MultinomialNB_Parameters.json',
 'LOG_FILE': 'MultinomialNB_RunLog.txt',
 'CHECK_TRAINING_SCORE': False,
 'EXPORT_SUBMISSION': True,
 'STAMP_PREDICTION_FILE': True,
 'LOG_RUN': True,
 'SKIP_IF_ALREADY_LOGGED': True,
 'USE_SAFE_TEXT_CLEANING': True,
 'STRIP_LATEX_MATH': True,
 'STRIP_URLS': True,
 'NORMALIZE_WHITESPACE': True,
 'VAL_FRACTION': 0.2,
 'RANDOM_STATE': 42,
 'MAX_FEATURES': 750000,
 'MIN_DF': 2,
 'MAX_DF': 0.85,
 'STOP_WORDS': 'english',
 'NGRAM_RANGE': (1, 2),
 'SUBLINEAR_TF': False,
 'MODEL': 'multinomial',
 'NORM': False,
 'ALPHA': 0.003,
 'FIT_PRIOR': False}

## **NEW:** Parameter Tuning Recollection
Tuning first consisted of a sweep of the parameters using a broad range across most, but not all variables. A sweep script using iteration was used to try many values/permutations, and high performing ones would be tested again using 3-folds. It bruteforced a bunch of ngram (1,1; 1,2), alpha (0.1-1.0), max/min_df (0.5-1.0 and 2-5 words respectively) combinations. None of these yieleded great results, so I stochastically (arbitrarily) expanded the bounds of alpha and got better results specifically with bigrams. This lead to finding a monotonic decrease until landing on the current alpha. Max features was also increased, and was found to have minimal effect after 1 million. It seems like having too high of the smoothing parameter caused too great of initial weight which had more gravity with a larger feature count.

- At `mf=5000`, the best `alpha` was around `0.01` (0.5845).
- At `mf=1M`, the best `alpha` dropped to `0.001` (0.6512).
- At `mf=2M` with `sublinear=True`, `0.002` won (0.6513).
- At `mf=750k` with `sublinear=False`, `0.003` won (0.6549).

With sublinear=False the counts are larger (not log-dampened), so a slightly larger `alpha` is needed to avoid over-trusting rare features.


When `ngram=(1,1)` was tried against the prevailing `(1,2)` winner,
macro-F1 dropped by 0.01–0.02. `(1,3)` was tried briefly and also didn't
help, tried with multiple feature counts, lower and higher.

For stop words:
`'english'` beat `None` by roughly 0.01–0.03 at every tested `alpha` and
`max_features`. Leaving it as `None` forced the model to spend vocabulary
budget on function words that are class-independent. Didn't touch much.


## Helpers

Pure functions: text cleaner to replace latex and urls with distinct words 'URLHERE' and whatnot. Run-id just labels the test/training run, submission path builder, and the JSONL run-log dedup helpers make sure that if I actually already tried an identical set of parameters it wouldn't bother retraining and running again.

They all read from the `CONFIG` dict above so behaviour stays in sync with the config cell.

In [76]:
_LATEX_MATH_RE = re.compile(r'\$.*?\$')
_URL_RE = re.compile(r'http\S+')
_WHITESPACE_RE = re.compile(r'\s+')


def safe_clean_arxiv_text(text):
    if CONFIG['STRIP_LATEX_MATH']:
        text = _LATEX_MATH_RE.sub('LATEX_MATH_HERE', text)
    if CONFIG['STRIP_URLS']:
        text = _URL_RE.sub('URL_HERE', text)
    if CONFIG['NORMALIZE_WHITESPACE']:
        text = _WHITESPACE_RE.sub(' ', text).strip()
    return text


def get_editable_parameters():
    """Return the same JSON-serialisable parameter dict as the .py script."""
    params = dict(CONFIG)
    params['NGRAM_RANGE'] = list(CONFIG['NGRAM_RANGE'])
    params['RUN_PIPELINE'] = True
    params['EXPORT_PARAMETERS_ONLY'] = False
    return params


_DEDUP_KEYS = (
    'USE_SAFE_TEXT_CLEANING',
    'STRIP_LATEX_MATH',
    'STRIP_URLS',
    'NORMALIZE_WHITESPACE',
    'VAL_FRACTION',
    'RANDOM_STATE',
    'MAX_FEATURES',
    'MIN_DF',
    'MAX_DF',
    'STOP_WORDS',
    'NGRAM_RANGE',
    'SUBLINEAR_TF',
    'MODEL',
    'NORM',
    'ALPHA',
    'FIT_PRIOR',
)


def make_classifier():
    """Instantiate the NB estimator chosen by CONFIG['MODEL'].

    Centralised so the validation cell and the full-fit cell stay in sync, and
    so adding new model variants in future is one edit instead of two.
    """
    model = CONFIG['MODEL']
    if model == 'multinomial':
        return MultinomialNB(alpha=CONFIG['ALPHA'], fit_prior=CONFIG['FIT_PRIOR'])
    if model == 'complement':
        return ComplementNB(
            alpha=CONFIG['ALPHA'],
            fit_prior=CONFIG['FIT_PRIOR'],
            norm=CONFIG['NORM'],
        )
    raise ValueError(f"Unknown MODEL: {model!r}. Use 'multinomial' or 'complement'.")


def _params_signature(params):
    return tuple(str(params.get(key)) for key in _DEDUP_KEYS)


def find_logged_run(params=None):
    if params is None:
        params = get_editable_parameters()
    log_file = CONFIG['LOG_FILE']
    if not os.path.exists(log_file):
        return None
    target = _params_signature(params)
    match = None
    with open(log_file, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                continue
            if _params_signature(entry.get('parameters', {})) == target:
                match = entry
    return match


def make_run_id():
    return datetime.now().strftime('%Y%m%dT%H%M%S')


def build_submission_path(run_id):
    if not CONFIG['STAMP_PREDICTION_FILE']:
        return CONFIG['OUTPUT_FILE']
    base, ext = os.path.splitext(CONFIG['OUTPUT_FILE'])
    return f'{base}_{run_id}{ext}'


def append_run_log(run_id, val_macro_f1, train_accuracy, prediction_file):
    payload = {
        'run_id': run_id,
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'val_macro_f1': val_macro_f1,
        'train_accuracy': train_accuracy,
        'prediction_file': prediction_file,
        'parameters': get_editable_parameters(),
    }
    with open(CONFIG['LOG_FILE'], 'a', encoding='utf-8') as file:
        file.write(json.dumps(payload) + '\n')
    print(f"Appended run summary to {CONFIG['LOG_FILE']} (run_id={run_id})")

## Load data (run once per kernel)

The `if 'train_df' not in globals() or FORCE_RELOAD` guard makes this cell a no-op on subsequent re-executions, so you can safely *Run All* without paying for a full CSV reparse. To force a fresh load, either set `FORCE_RELOAD = True` here or `del train_df, test_df` from another cell.

In [77]:
FORCE_RELOAD = False

if FORCE_RELOAD or 'train_df' not in globals() or 'test_df' not in globals():
    print('Reading train.csv and test.csv ...')
    train_df = pd.read_csv(f"{CONFIG['DATA_DIR']}train.csv", sep='\t')
    test_df = pd.read_csv(f"{CONFIG['DATA_DIR']}test.csv", sep='\t')
    print(f'  train_df: {train_df.shape}')
    print(f'  test_df : {test_df.shape}')
else:
    print('Reusing cached train_df / test_df already in memory.')

X_text_raw = train_df['abstract'].fillna('').astype(str)
y = train_df['label_id'].astype(int)
X_test_text_raw = test_df['abstract'].fillna('').astype(str)

X_text_raw.head(2)

Reusing cached train_df / test_df already in memory.


0    Project-based learning plays a crucial role in...
1      Edge computing is a distributed computing pa...
Name: abstract, dtype: object

## Clean text

Re-run after toggling any of `USE_SAFE_TEXT_CLEANING`, `STRIP_LATEX_MATH`, `STRIP_URLS`, `NORMALIZE_WHITESPACE`. Output binds `X_text` / `X_test_text`, which the modeling cells consume.

In [78]:
cleaning_active = CONFIG['USE_SAFE_TEXT_CLEANING'] and (
    CONFIG['STRIP_LATEX_MATH']
    or CONFIG['STRIP_URLS']
    or CONFIG['NORMALIZE_WHITESPACE']
)

if cleaning_active:
    print('Applying safe_clean_arxiv_text ...')
    X_text = X_text_raw.apply(safe_clean_arxiv_text)
    X_test_text = X_test_text_raw.apply(safe_clean_arxiv_text)
else:
    print('Cleaning disabled; using raw abstracts.')
    X_text = X_text_raw
    X_test_text = X_test_text_raw

X_text.head(2)

Applying safe_clean_arxiv_text ...


0    Project-based learning plays a crucial role in...
1    Edge computing is a distributed computing para...
Name: abstract, dtype: object

## Dedup check (optional)

Mirrors `SKIP_IF_ALREADY_LOGGED` from the script. Just prints the matching prior run if any; it does **not** abort downstream cells (in a notebook you usually want to see numbers anyway).

In [79]:
if CONFIG['SKIP_IF_ALREADY_LOGGED']:
    prior = find_logged_run()
    if prior is not None:
        prior_f1 = prior.get('val_macro_f1')
        prior_acc = prior.get('train_accuracy')
        prior_run_id = prior.get('run_id') or prior.get('timestamp')
        prior_pred = prior.get('prediction_file') or 'none'
        f1_str = f'{prior_f1:.4f}' if isinstance(prior_f1, (int, float)) else 'n/a'
        acc_str = f'{prior_acc:.4f}' if isinstance(prior_acc, (int, float)) else 'n/a'
        print(
            'Matching parameter set already logged:\n'
            f'  prior run_id        : {prior_run_id}\n'
            f'  prior val_macro_f1  : {f1_str}\n'
            f'  prior train_accuracy: {acc_str}\n'
            f'  prior prediction    : {prior_pred}'
        )
    else:
        print('No prior log entry matches the current CONFIG.')
else:
    print('SKIP_IF_ALREADY_LOGGED is False; skipping dedup check.')

No prior log entry matches the current CONFIG.


## Validate (held-out split)

Skipped when `VAL_FRACTION == 0`. Builds a *separate* TF-IDF vocabulary from the train split only, so validation numbers don't leak the val rows into vectorizer fitting.

In [80]:
val_macro_f1 = None

if CONFIG['VAL_FRACTION'] > 0:
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_text,
        y,
        test_size=CONFIG['VAL_FRACTION'],
        random_state=CONFIG['RANDOM_STATE'],
        stratify=y,
    )
    vec_val = TfidfVectorizer(
        max_features=CONFIG['MAX_FEATURES'],
        min_df=CONFIG['MIN_DF'],
        max_df=CONFIG['MAX_DF'],
        stop_words=CONFIG['STOP_WORDS'],
        ngram_range=CONFIG['NGRAM_RANGE'],
        sublinear_tf=CONFIG['SUBLINEAR_TF'],
    )
    X_tr_vec = vec_val.fit_transform(X_tr)

    clf_val = make_classifier()
    clf_val.fit(X_tr_vec, y_tr)

    y_hat = clf_val.predict(vec_val.transform(X_va))
    val_macro_f1 = float(f1_score(y_va, y_hat, average='macro'))
    print(f'Validation macro-F1: {val_macro_f1:.4f}')
else:
    print('VAL_FRACTION is 0; skipping validation.')

Validation macro-F1: 0.6549


## Full fit + (optional) submission

Refits TF-IDF + the chosen Naive Bayes model (`MODEL` in `CONFIG`, either `'multinomial'` or `'complement'`) on the entire training set, then optionally scores on train and writes a Kaggle CSV. Skipped entirely when both `CHECK_TRAINING_SCORE` and `EXPORT_SUBMISSION` are False.

In [81]:
run_id = make_run_id()
train_accuracy = None
prediction_file = None

needs_full_fit = CONFIG['EXPORT_SUBMISSION'] or CONFIG['CHECK_TRAINING_SCORE']

if needs_full_fit:
    vectorizer = TfidfVectorizer(
        max_features=CONFIG['MAX_FEATURES'],
        min_df=CONFIG['MIN_DF'],
        max_df=CONFIG['MAX_DF'],
        stop_words=CONFIG['STOP_WORDS'],
        ngram_range=CONFIG['NGRAM_RANGE'],
        sublinear_tf=CONFIG['SUBLINEAR_TF'],
    )
    X_full_vec = vectorizer.fit_transform(X_text)

    clf = make_classifier()
    clf.fit(X_full_vec, y)

    if CONFIG['CHECK_TRAINING_SCORE']:
        train_accuracy = float(clf.score(X_full_vec, y))
        print(f'Training accuracy: {train_accuracy:.4f}')

    if CONFIG['EXPORT_SUBMISSION']:
        X_test_vec = vectorizer.transform(X_test_text)
        test_preds = clf.predict(X_test_vec)

        submission = pd.DataFrame({
            'id': test_df['id'].astype(int),
            'label_id': test_preds.astype(int),
        })
        prediction_file = build_submission_path(run_id)
        submission.to_csv(prediction_file, index=False)
        print(f'Saved Kaggle submission to {prediction_file} (run_id={run_id})')
else:
    print('Skipped full retrain (EXPORT_SUBMISSION and CHECK_TRAINING_SCORE are False).')

Saved Kaggle submission to MultinomialNB_Prediction_20260420T161337.csv (run_id=20260420T161337)


## Log run

Appends one JSONL line to `MultinomialNB_RunLog.txt` containing the `run_id`, val score, train accuracy, prediction filename, and the full editable-parameter snapshot. Skip by setting `LOG_RUN` to False in `CONFIG`.

In [82]:
if CONFIG['LOG_RUN']:
    append_run_log(run_id, val_macro_f1, train_accuracy, prediction_file)
else:
    print('LOG_RUN is False; not writing to run log.')

Appended run summary to MultinomialNB_RunLog.txt (run_id=20260420T161337)


## Export parameters JSON (optional)

Equivalent of the `.py` script's `EXPORT_PARAMETERS_ONLY` short-circuit: write the current editable parameters to `MultinomialNB_Parameters.json`.

In [83]:
with open(CONFIG['PARAMETERS_FILE'], 'w', encoding='utf-8') as file:
    json.dump(get_editable_parameters(), file, indent=2)
print(f"Saved editable parameters to {CONFIG['PARAMETERS_FILE']}")

Saved editable parameters to MultinomialNB_Parameters.json


"""
Task 3 sweep harness.

Loads the training data and pre-cleans it once, then iterates over TF-IDF
configurations in the outer loop and Naive Bayes configurations in the inner
loop, writing a row per permutation to a CSV.

Supports two evaluation modes:
- Single split (N_FOLDS = 1): fast, default for wide sweeps.
- Stratified k-fold (N_FOLDS > 1): slower, more stable rankings.

Use `revalidate_top()` to take the best configs from a single-split sweep and
re-evaluate them with k-fold CV for confirmation.

Designed so you can also import these helpers from a Jupyter notebook and reuse
the cached data across many sweeps without reloading.

Dependencies: pandas, scikit-learn
"""

import ast
import csv
import math
import os
import re
import time
from datetime import datetime

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB

# ==========================================
# HARDCODED CONFIGURATION
# ==========================================

# Paths
DATA_DIR = './'
RESULTS_FILE = 'MultinomialNB_Sweep_Results.csv'
CV_RESULTS_FILE = 'MultinomialNB_CV_Results.csv'

# Validation split (used when N_FOLDS == 1)
VAL_FRACTION = 0.2
RANDOM_STATE = 42

# Evaluation mode
# 1 = single stratified train/val split (fast)
# >1 = stratified k-fold CV (more stable, k* slower)
#
# Pinned to 1 so the sweep uses the same split (VAL_FRACTION=0.2,
# RANDOM_STATE=42) as task3_multinomial_nb.py. That makes the val_macro_f1
# column directly comparable to entries in MultinomialNB_RunLog.txt, including
# the current #1 at f1=0.6512, rather than being an independent CV mean.
# AUTO_REVALIDATE (below) will then re-score top hits with k-fold CV.
N_FOLDS = 1

# Cleaning options to sweep over (True = use safe_clean_arxiv_text, False = raw)
# Pinned to True: every top-5 run-log entry used clean=True.
CLEANING_GRID = [True]

# ------------------------------------------------------------------------------
# Targeted neighborhood sweep around the current #1 run-log result:
#     max_features=1_000_000, min_df=2, max_df=0.85, ngram=(1, 2),
#     stop_words='english', sublinear_tf=True, clean=True,
#     model='multinomial', alpha=0.001, fit_prior=False  -> f1=0.6512
#
# Only the three axes most likely to improve on #1 are varied. Everything else
# is pinned to the known-good value. The previous broader grid is preserved
# below (commented) so you can revert.
#
# Permutation count: 3 (max_features) * 3 (max_df) * 3 (alpha) = 27
# ------------------------------------------------------------------------------
TFIDF_GRID = [
    {
        'max_features': max_features,
        'min_df': 2,
        'max_df': max_df,
        'stop_words': 'english',
        'ngram_range': (1, 3),
        'sublinear_tf': False,
    }
    for max_features in (500_000, 650_000, 750_000, 850_000, 1_000_000, 1_500_000)
    for max_df in (0.85,)
]

# Inner loop: Naive Bayes configurations.
#
# Each dict must include 'model' ('multinomial' or 'complement'). MNB respects
# 'fit_prior'; CNB ignores fit_prior and instead respects 'norm'. The harness
# silently drops irrelevant keys before passing kwargs to the estimator.
#
# CNB is disabled for the neighborhood sweep: best CNB run-log entry so far is
# f1~0.52, well below the MNB top. fit_prior=False only, matching #1.
INCLUDE_MULTINOMIAL_NB = True
INCLUDE_COMPLEMENT_NB = False

_MNB_CONFIGS = [
    {'model': 'multinomial', 'alpha': alpha, 'fit_prior': False, 'norm': False}
    for alpha in (0.0005, 0.00075, 0.001, 0.0015, 0.002, 0.0025, 0.003)
]

_CNB_CONFIGS = [
    {'model': 'complement', 'alpha': alpha, 'fit_prior': True, 'norm': norm}
    for alpha in (0.0001, 0.001, 0.005, 0.01, 0.02, 0.03, 0.04, 0.1, 0.3)
    for norm in (False, True)
]

# ------------------------------------------------------------------------------
# Previous broader grid (kept for reference; re-enable by uncommenting these
# blocks and commenting out the neighborhood blocks above):
#
# N_FOLDS = 2
# TFIDF_GRID = [
#     {
#         'max_features': max_features,
#         'min_df': min_df,
#         'max_df': max_df,
#         'stop_words': 'english',
#         'ngram_range': ngram_range,
#         'sublinear_tf': True,
#     }
#     for max_features in (100000, 200000, 500000, 1000000, 2000000)
#     for min_df in (1, 2, 3, 5)
#     for max_df in (0.3, 0.5, 0.85, 0.95)
#     for ngram_range in ((1, 1), (1, 2))
# ]
# INCLUDE_COMPLEMENT_NB = True
# _MNB_CONFIGS = [
#     {'model': 'multinomial', 'alpha': alpha, 'fit_prior': False, 'norm': False}
#     for alpha in (0.0001, 0.001, 0.005, 0.01, 0.03)
# ]
# ------------------------------------------------------------------------------

NB_GRID = (
    (_MNB_CONFIGS if INCLUDE_MULTINOMIAL_NB else [])
    + (_CNB_CONFIGS if INCLUDE_COMPLEMENT_NB else [])
)

# Re-validation defaults
# Threshold tuned around the current #1 (single-split f1=0.6512). Anything
# that beats or matches that on the single split gets promoted to k-fold CV.
REVALIDATE_TOP_N = 3
REVALIDATE_N_FOLDS = 3
REVALIDATE_THRESHOLD = 0.65               # Promotion bar: single-split f1 must >= this
CV_PASS_THRESHOLD = 0.65                  # CV pass bar: k-fold mean f1 must >= this to count
                                          # as PASS; below it is marked REGRESSED. A small
                                          # margin below the promotion bar accounts for CV
                                          # typically scoring slightly lower than the tuned
                                          # single split.
AUTO_REVALIDATE = True                    # After a single-split sweep, auto re-validate
                                          # configs scoring >= REVALIDATE_THRESHOLD with CV

# ==========================================

# Precompiled regexes for the cleaner
_LATEX_MATH_RE = re.compile(r'\$.*?\$')
_URL_RE = re.compile(r'http\S+')
_WHITESPACE_RE = re.compile(r'\s+')


def clean_text(text):
    text = _LATEX_MATH_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)
    text = _WHITESPACE_RE.sub(' ', text).strip()
    return text


def load_data(data_dir=DATA_DIR, val_fraction=VAL_FRACTION, random_state=RANDOM_STATE):
    """
    Load train.csv once, build cleaned and raw versions, and stratify-split
    each into a fixed (train, val) using the same seed.

    Returns a dict keyed by `use_clean` -> {'text': Series, 'y': Series,
    'split': (X_tr, X_va, y_tr, y_va)}.
    """
    print('Loading train.csv ...')
    train = pd.read_csv(
        f'{data_dir}train.csv',
        sep='\t',
        usecols=['abstract', 'label_id'],
    )

    raw_text = train['abstract'].fillna('').astype(str)
    y = train['label_id'].astype(int)

    print('Cleaning text once ...')
    cleaned_text = raw_text.apply(clean_text)

    text_versions = {False: raw_text, True: cleaned_text}

    data = {}
    for use_clean, text in text_versions.items():
        X_tr, X_va, y_tr, y_va = train_test_split(
            text,
            y,
            test_size=val_fraction,
            random_state=random_state,
            stratify=y,
        )
        data[use_clean] = {
            'text': text,
            'y': y,
            'split': (X_tr, X_va, y_tr, y_va),
        }
    return data


# Fields that uniquely identify an evaluated permutation in the CSV. `model` and
# `norm` were added when ComplementNB was introduced; old rows (which were all
# MultinomialNB) are migrated to model='multinomial', norm='False' by
# `_ensure_schema` so they keep matching their original signatures.
_SIGNATURE_FIELDS = (
    'use_safe_text_cleaning',
    'max_features', 'min_df', 'max_df', 'stop_words', 'ngram_range', 'sublinear_tf',
    'model', 'alpha', 'fit_prior', 'norm',
    'n_folds',
)

_CSV_FIELDNAMES = [
    'timestamp',
    'source',                       # 'sweep' or 'revalidation'
    'use_safe_text_cleaning',
    'max_features', 'min_df', 'max_df', 'stop_words', 'ngram_range', 'sublinear_tf',
    'model',                        # 'multinomial' or 'complement'
    'alpha', 'fit_prior', 'norm',   # fit_prior is for MNB only, norm is for CNB only
    'n_folds',
    'val_macro_f1', 'val_macro_f1_std', 'fit_seconds',
    # Annotation fields. Populated for revalidation rows directly, and back-filled
    # onto matching sweep rows so the sweep CSV becomes a single highlight view.
    'original_val_macro_f1',        # the single-split score that triggered promotion
    'delta_vs_original',            # cv_mean - original (positive = CV agrees / improved)
    'cv_status',                    # '', 'PASS', or 'REGRESSED'
    'cv_macro_f1',                  # only set on sweep rows after annotation
]


def _nb_field(nb_kwargs, key, default):
    """Read a NB hyperparameter, falling back to a default if absent.

    Keeps `_signature` and `_row_for_writer` simple while letting individual NB
    dicts omit keys that don't apply to their model (e.g. CNB rows can omit
    `fit_prior`, MNB rows can omit `norm`).
    """
    value = nb_kwargs.get(key, default)
    return default if value is None else value


def _signature(use_clean, tfidf_kwargs, nb_kwargs, n_folds):
    """Build a hashable signature that uniquely identifies a permutation."""
    return (
        str(use_clean),
        str(tfidf_kwargs['max_features']),
        str(tfidf_kwargs['min_df']),
        str(tfidf_kwargs['max_df']),
        str(tfidf_kwargs['stop_words']),
        str(tfidf_kwargs['ngram_range']),
        str(tfidf_kwargs['sublinear_tf']),
        str(_nb_field(nb_kwargs, 'model', 'multinomial')),
        str(nb_kwargs['alpha']),
        str(_nb_field(nb_kwargs, 'fit_prior', True)),
        str(_nb_field(nb_kwargs, 'norm', False)),
        str(n_folds),
    )


def _load_completed_signatures(results_file):
    """Read the existing results CSV (if any) and return the set of signatures
    that have already been evaluated."""
    if not os.path.exists(results_file):
        return set()
    completed = set()
    with open(results_file, 'r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        for row in reader:
            try:
                completed.add(tuple(str(row.get(field, '')) for field in _SIGNATURE_FIELDS))
            except KeyError:
                continue
    return completed


def _ensure_schema(path):
    """Migrate an existing CSV to the current `_CSV_FIELDNAMES`.

    If the file's header is outdated (added columns since it was first written),
    re-write it once with the new header so subsequent appends stay aligned.
    Existing rows have missing columns back-filled with empty strings; nothing
    is dropped or modified beyond shape.
    """
    if not os.path.exists(path):
        return
    with open(path, 'r', newline='', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        existing = list(reader.fieldnames or [])
        if existing == _CSV_FIELDNAMES:
            return
        rows = list(reader)

    # Defaults applied to old rows for newly added columns. Anything not listed
    # here is back-filled with empty string. These specific defaults preserve
    # signature identity for legacy rows: every previously-recorded run was a
    # MultinomialNB with fit_prior controlled per-row, so model and norm get
    # the values that match what those rows would have produced under the new
    # signature scheme.
    legacy_defaults = {
        'model': 'multinomial',
        'norm': 'False',
    }

    with open(path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=_CSV_FIELDNAMES)
        writer.writeheader()
        for row in rows:
            new_row = {}
            for field in _CSV_FIELDNAMES:
                if field in row and row[field] != '':
                    new_row[field] = row[field]
                else:
                    new_row[field] = legacy_defaults.get(field, '')
            writer.writerow(new_row)
    added = [f for f in _CSV_FIELDNAMES if f not in existing]
    print(f'Migrated {path} schema (added columns: {added or "none"}).')


def _make_classifier(nb_kwargs):
    """Dispatch on `nb_kwargs['model']` and return a fresh estimator instance.

    Only the keys that the chosen estimator actually accepts are passed through.
    This lets the NB grid carry a uniform shape (every dict has model/alpha/
    fit_prior/norm) without sklearn complaining about unknown kwargs.
    """
    model = _nb_field(nb_kwargs, 'model', 'multinomial')
    if model == 'multinomial':
        return MultinomialNB(
            alpha=float(nb_kwargs['alpha']),
            fit_prior=bool(_nb_field(nb_kwargs, 'fit_prior', True)),
        )
    if model == 'complement':
        return ComplementNB(
            alpha=float(nb_kwargs['alpha']),
            fit_prior=bool(_nb_field(nb_kwargs, 'fit_prior', True)),
            norm=bool(_nb_field(nb_kwargs, 'norm', False)),
        )
    raise ValueError(f"Unknown NB model: {model!r}. Use 'multinomial' or 'complement'.")


def _row_for_writer(
    use_clean, tfidf_kwargs, nb_kwargs, n_folds,
    val_macro_f1, val_macro_f1_std, fit_seconds,
    source='sweep', original_val_macro_f1=None, cv_status='', cv_macro_f1=None,
):
    if original_val_macro_f1 is not None:
        delta = round(val_macro_f1 - float(original_val_macro_f1), 4)
    else:
        delta = ''
    return {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'source': source,
        'use_safe_text_cleaning': use_clean,
        'max_features': tfidf_kwargs['max_features'],
        'min_df': tfidf_kwargs['min_df'],
        'max_df': tfidf_kwargs['max_df'],
        'stop_words': tfidf_kwargs['stop_words'],
        'ngram_range': str(tfidf_kwargs['ngram_range']),
        'sublinear_tf': tfidf_kwargs['sublinear_tf'],
        'model': _nb_field(nb_kwargs, 'model', 'multinomial'),
        'alpha': nb_kwargs['alpha'],
        'fit_prior': _nb_field(nb_kwargs, 'fit_prior', True),
        'norm': _nb_field(nb_kwargs, 'norm', False),
        'n_folds': n_folds,
        'val_macro_f1': val_macro_f1,
        'val_macro_f1_std': '' if val_macro_f1_std is None else val_macro_f1_std,
        'fit_seconds': round(fit_seconds, 3),
        'original_val_macro_f1': '' if original_val_macro_f1 is None else original_val_macro_f1,
        'delta_vs_original': delta,
        'cv_status': cv_status,
        'cv_macro_f1': '' if cv_macro_f1 is None else cv_macro_f1,
    }


def _evaluate_single_split(data_entry, tfidf_kwargs, nb_grid, completed, use_clean, results_writer, results_file_handle, log_prefix):
    """Run all NB configs against one (X_tr, X_va, y_tr, y_va) split."""
    X_tr, X_va, y_tr, y_va = data_entry['split']

    pending = [
        nb_kwargs
        for nb_kwargs in nb_grid
        if _signature(use_clean, tfidf_kwargs, nb_kwargs, 1) not in completed
    ]
    if not pending:
        print(f'{log_prefix} skip vectorizer (all NB permutations already done)')
        return 0, len(nb_grid)

    vec = TfidfVectorizer(**tfidf_kwargs)
    Xtr_vec = vec.fit_transform(X_tr)
    Xva_vec = vec.transform(X_va)

    written = 0
    skipped = 0
    for nb_kwargs in nb_grid:
        sig = _signature(use_clean, tfidf_kwargs, nb_kwargs, 1)
        if sig in completed:
            skipped += 1
            continue

        start = time.perf_counter()
        clf = _make_classifier(nb_kwargs).fit(Xtr_vec, y_tr)
        y_hat = clf.predict(Xva_vec)
        elapsed = time.perf_counter() - start

        val_f1 = float(f1_score(y_va, y_hat, average='macro'))
        row = _row_for_writer(
            use_clean, tfidf_kwargs, nb_kwargs, 1,
            val_f1, None, elapsed,
        )
        results_writer.writerow(row)
        results_file_handle.flush()
        completed.add(sig)
        written += 1

        print(
            f'{log_prefix} f1={val_f1:.4f} '
            f'model={_nb_field(nb_kwargs, "model", "multinomial")} '
            f'alpha={nb_kwargs["alpha"]} '
            f'fit_prior={_nb_field(nb_kwargs, "fit_prior", True)} '
            f'norm={_nb_field(nb_kwargs, "norm", False)}'
        )
    return written, skipped


def _evaluate_kfold(
    data_entry, tfidf_kwargs, nb_grid, completed, use_clean, n_folds, random_state,
    results_writer, results_file_handle, log_prefix,
    source='sweep', original_score=None, cv_pass_threshold=None,
):
    """Run all NB configs against `n_folds` stratified folds, fitting TF-IDF
    once per fold and reusing the cached matrices for every NB config.

    `source`, `original_score`, and `cv_pass_threshold` are only meaningful when
    this is invoked from `revalidate_top()`; they let us tag each CV row with
    the single-split score that triggered it and a PASS/REGRESSED status.
    """
    text = data_entry['text']
    y = data_entry['y']

    pending = [
        nb_kwargs
        for nb_kwargs in nb_grid
        if _signature(use_clean, tfidf_kwargs, nb_kwargs, n_folds) not in completed
    ]
    if not pending:
        print(f'{log_prefix} skip vectorizer (all NB permutations already done)')
        return 0, len(nb_grid)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    fold_caches = []
    for tr_idx, va_idx in skf.split(text, y):
        X_tr_text = text.iloc[tr_idx]
        X_va_text = text.iloc[va_idx]
        y_tr = y.iloc[tr_idx]
        y_va = y.iloc[va_idx]
        vec = TfidfVectorizer(**tfidf_kwargs)
        Xtr_vec = vec.fit_transform(X_tr_text)
        Xva_vec = vec.transform(X_va_text)
        fold_caches.append((Xtr_vec, y_tr, Xva_vec, y_va))

    written = 0
    skipped = 0
    for nb_kwargs in nb_grid:
        sig = _signature(use_clean, tfidf_kwargs, nb_kwargs, n_folds)
        if sig in completed:
            skipped += 1
            continue

        start = time.perf_counter()
        fold_f1s = []
        for Xtr_vec, y_tr, Xva_vec, y_va in fold_caches:
            clf = _make_classifier(nb_kwargs).fit(Xtr_vec, y_tr)
            y_hat = clf.predict(Xva_vec)
            fold_f1s.append(float(f1_score(y_va, y_hat, average='macro')))
        elapsed = time.perf_counter() - start

        mean_f1 = sum(fold_f1s) / len(fold_f1s)
        var = sum((s - mean_f1) ** 2 for s in fold_f1s) / len(fold_f1s)
        std_f1 = math.sqrt(var)

        cv_status = ''
        if cv_pass_threshold is not None:
            cv_status = 'PASS' if mean_f1 >= cv_pass_threshold else 'REGRESSED'

        row = _row_for_writer(
            use_clean, tfidf_kwargs, nb_kwargs, n_folds,
            mean_f1, round(std_f1, 4), elapsed,
            source=source,
            original_val_macro_f1=original_score,
            cv_status=cv_status,
        )
        results_writer.writerow(row)
        results_file_handle.flush()
        completed.add(sig)
        written += 1

        status_tag = f' [{cv_status}]' if cv_status else ''
        print(
            f'{log_prefix} f1={mean_f1:.4f} (+/- {std_f1:.4f}){status_tag} '
            f'model={_nb_field(nb_kwargs, "model", "multinomial")} '
            f'alpha={nb_kwargs["alpha"]} '
            f'fit_prior={_nb_field(nb_kwargs, "fit_prior", True)} '
            f'norm={_nb_field(nb_kwargs, "norm", False)}'
        )
    return written, skipped


def run_sweep(
    data,
    cleaning_grid=None,
    tfidf_grid=None,
    nb_grid=None,
    results_file=RESULTS_FILE,
    n_folds=N_FOLDS,
    random_state=RANDOM_STATE,
):
    """
    Run a full sweep against the supplied `data` dict (from `load_data()`).

    Single-split mode (n_folds == 1): one TF-IDF fit per outer config, cheap NB
    fits inside.

    K-fold mode (n_folds >= 2): k TF-IDF fits per outer config, then NB grid
    runs against all k cached matrices. Logs mean and std of macro-F1.

    Resumable: on restart, any permutation whose signature is already present
    in `results_file` is skipped. Each row is flushed on write so partial runs
    are preserved if you cancel.
    """
    cleaning_grid = CLEANING_GRID if cleaning_grid is None else cleaning_grid
    tfidf_grid = TFIDF_GRID if tfidf_grid is None else tfidf_grid
    nb_grid = NB_GRID if nb_grid is None else nb_grid

    if n_folds < 1:
        raise ValueError(f'n_folds must be >= 1 (got {n_folds})')

    _ensure_schema(results_file)
    write_header = not os.path.exists(results_file)
    completed = _load_completed_signatures(results_file)

    total = len(cleaning_grid) * len(tfidf_grid) * len(nb_grid)
    already_done = sum(
        1
        for use_clean in cleaning_grid
        for tfidf_kwargs in tfidf_grid
        for nb_kwargs in nb_grid
        if _signature(use_clean, tfidf_kwargs, nb_kwargs, n_folds) in completed
    )
    remaining = total - already_done
    print(
        f'Sweep size: {total} permutations, n_folds={n_folds} '
        f'({already_done} already done, {remaining} to run)'
    )
    if remaining == 0:
        print('Nothing to do. Delete or move the results file to start over.')
        return

    counter = 0
    skipped = 0
    with open(results_file, 'a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=_CSV_FIELDNAMES)
        if write_header:
            writer.writeheader()

        for use_clean in cleaning_grid:
            for tfidf_kwargs in tfidf_grid:
                log_prefix = (
                    f'[clean={use_clean} mf={tfidf_kwargs["max_features"]} '
                    f'mindf={tfidf_kwargs["min_df"]} '
                    f'ngram={tfidf_kwargs["ngram_range"]}]'
                )
                if n_folds == 1:
                    written, skip = _evaluate_single_split(
                        data[use_clean], tfidf_kwargs, nb_grid, completed,
                        use_clean, writer, file, log_prefix,
                    )
                else:
                    written, skip = _evaluate_kfold(
                        data[use_clean], tfidf_kwargs, nb_grid, completed,
                        use_clean, n_folds, random_state,
                        writer, file, log_prefix,
                    )
                counter += written
                skipped += skip

    print(
        f'Done. Wrote {counter} new rows to {results_file} '
        f'(skipped {skipped} already-done permutations).'
    )

    if n_folds == 1 and AUTO_REVALIDATE:
        print(
            f'AUTO_REVALIDATE on: promoting configs with val_macro_f1 >= '
            f'{REVALIDATE_THRESHOLD} to {REVALIDATE_N_FOLDS}-fold CV ...'
        )
        revalidate_top(
            data,
            top_n=None,
            n_folds=REVALIDATE_N_FOLDS,
            min_score=REVALIDATE_THRESHOLD,
            source_results_file=results_file,
            random_state=random_state,
            cv_pass_threshold=CV_PASS_THRESHOLD,
        )


def _row_to_kwargs(row):
    """Parse a CSV row from RESULTS_FILE back into (use_clean, tfidf_kwargs, nb_kwargs)."""
    use_clean = str(row['use_safe_text_cleaning']).lower() in ('true', '1')
    stop_words_raw = row['stop_words']
    stop_words = None if str(stop_words_raw).lower() in ('', 'none', 'nan') else str(stop_words_raw)
    tfidf_kwargs = {
        'max_features': int(row['max_features']),
        'min_df': int(row['min_df']),
        'max_df': float(row['max_df']),
        'stop_words': stop_words,
        'ngram_range': tuple(ast.literal_eval(row['ngram_range'])),
        'sublinear_tf': str(row['sublinear_tf']).lower() in ('true', '1'),
    }
    model_raw = str(row.get('model', '') or 'multinomial').lower()
    model = model_raw if model_raw in ('multinomial', 'complement') else 'multinomial'
    nb_kwargs = {
        'model': model,
        'alpha': float(row['alpha']),
        'fit_prior': str(row.get('fit_prior', 'True')).lower() in ('true', '1'),
        'norm': str(row.get('norm', 'False')).lower() in ('true', '1'),
    }
    return use_clean, tfidf_kwargs, nb_kwargs


def revalidate_top(
    data,
    top_n=REVALIDATE_TOP_N,
    n_folds=REVALIDATE_N_FOLDS,
    min_score=None,
    source_results_file=RESULTS_FILE,
    cv_results_file=CV_RESULTS_FILE,
    sort_by='val_macro_f1',
    random_state=RANDOM_STATE,
    cv_pass_threshold=CV_PASS_THRESHOLD,
):
    """
    Read `source_results_file` and re-evaluate selected configs with `n_folds`
    stratified CV. Selection rules:

    - If `min_score` is set, only consider rows with `sort_by >= min_score`.
    - If `top_n` is set (and not None), then take at most that many of the
      remaining rows, sorted by `sort_by` descending.
    - Setting `top_n=None` and `min_score=0.65` revalidates *everything* above
      the threshold.

    Appends to `cv_results_file`. Resumable: configs already present there
    (matched on the full signature including `n_folds`) are skipped.
    """
    if not os.path.exists(source_results_file):
        print(f'{source_results_file} does not exist; nothing to revalidate.')
        return

    df = pd.read_csv(source_results_file)
    if df.empty:
        print(f'{source_results_file} is empty; nothing to revalidate.')
        return

    if min_score is not None:
        df = df[df[sort_by] >= min_score]
        if df.empty:
            print(f'No rows in {source_results_file} have {sort_by} >= {min_score}.')
            return

    df_sorted = df.sort_values(sort_by, ascending=False)
    if top_n is not None:
        df_sorted = df_sorted.head(top_n)

    print(
        f'Re-validating {len(df_sorted)} configs from {source_results_file} '
        f'with {n_folds}-fold CV '
        f'(min_score={min_score}, top_n={top_n}) ...'
    )

    _ensure_schema(cv_results_file)
    write_header = not os.path.exists(cv_results_file)
    completed = _load_completed_signatures(cv_results_file)

    written = 0
    skipped = 0
    with open(cv_results_file, 'a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=_CSV_FIELDNAMES)
        if write_header:
            writer.writeheader()

        for _, row in df_sorted.iterrows():
            use_clean, tfidf_kwargs, nb_kwargs = _row_to_kwargs(row)
            sig = _signature(use_clean, tfidf_kwargs, nb_kwargs, n_folds)
            if sig in completed:
                skipped += 1
                continue

            log_prefix = (
                f'[revalidate clean={use_clean} mf={tfidf_kwargs["max_features"]} '
                f'mindf={tfidf_kwargs["min_df"]} ngram={tfidf_kwargs["ngram_range"]}]'
            )
            sub_written, _ = _evaluate_kfold(
                data[use_clean], tfidf_kwargs, [nb_kwargs], completed,
                use_clean, n_folds, random_state,
                writer, file, log_prefix,
                source='revalidation',
                original_score=float(row[sort_by]),
                cv_pass_threshold=cv_pass_threshold,
            )
            written += sub_written

    print(
        f'Re-validation done. Wrote {written} new rows to {cv_results_file} '
        f'(skipped {skipped} already-validated configs).'
    )

    annotated = _annotate_sweep_with_revalidation_status(
        sweep_file=source_results_file,
        cv_file=cv_results_file,
    )
    _print_revalidation_summary(cv_results_file, cv_pass_threshold, annotated)


# Subset of signature fields used to match a sweep row to a CV row regardless
# of `n_folds` (because the sweep was n_folds=1 and the CV was n_folds=k).
_PARAM_ONLY_FIELDS = (
    'use_safe_text_cleaning',
    'max_features', 'min_df', 'max_df', 'stop_words', 'ngram_range', 'sublinear_tf',
    'model', 'alpha', 'fit_prior', 'norm',
)


def _param_key(row):
    """Build a `_PARAM_ONLY_FIELDS` tuple for matching across sweep and CV files."""
    return tuple(str(row.get(field, '')) for field in _PARAM_ONLY_FIELDS)


def _annotate_sweep_with_revalidation_status(sweep_file=RESULTS_FILE, cv_file=CV_RESULTS_FILE):
    """Back-fill `cv_macro_f1` and `cv_status` on sweep rows that have a
    matching CV result.

    Workflow:
    1. Load the CV file and build {param_key -> (best_cv_mean, best_cv_status)}.
       If the same params got revalidated more than once (e.g. different fold
       counts), keep the highest mean — that is the most generous read.
    2. Load the sweep file, fill the two columns for any matching row.
    3. Rewrite the sweep file with the full schema.

    Returns the number of sweep rows that were annotated.
    """
    if not os.path.exists(sweep_file) or not os.path.exists(cv_file):
        return 0

    cv_map = {}
    with open(cv_file, 'r', newline='', encoding='utf-8') as file:
        for row in csv.DictReader(file):
            try:
                cv_mean = float(row.get('val_macro_f1', ''))
            except (TypeError, ValueError):
                continue
            key = _param_key(row)
            cv_status = row.get('cv_status', '') or ''
            best = cv_map.get(key)
            if best is None or cv_mean > best[0]:
                cv_map[key] = (cv_mean, cv_status)

    if not cv_map:
        return 0

    with open(sweep_file, 'r', newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))

    annotated = 0
    for row in rows:
        key = _param_key(row)
        if key in cv_map:
            cv_mean, cv_status = cv_map[key]
            row['cv_macro_f1'] = cv_mean
            row['cv_status'] = cv_status
            annotated += 1

    with open(sweep_file, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=_CSV_FIELDNAMES)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: row.get(field, '') for field in _CSV_FIELDNAMES})

    print(f'Annotated {annotated} row(s) in {sweep_file} with cv_macro_f1 / cv_status.')
    return annotated


def _print_revalidation_summary(cv_file, cv_pass_threshold, annotated_count):
    """Print PASS/REGRESSED tallies and the top revalidated configs."""
    if not os.path.exists(cv_file):
        return
    df = pd.read_csv(cv_file)
    if df.empty:
        return

    reval = df[df.get('source', '') == 'revalidation'] if 'source' in df.columns else df
    if reval.empty:
        return

    pass_count = int((reval['cv_status'] == 'PASS').sum())
    regressed_count = int((reval['cv_status'] == 'REGRESSED').sum())

    print('--- Revalidation summary ---')
    print(
        f'Total revalidated: {len(reval)} | '
        f'PASS (>= {cv_pass_threshold}): {pass_count} | '
        f'REGRESSED: {regressed_count} | '
        f'Sweep rows highlighted: {annotated_count}'
    )

    top_cols = [
        'cv_status', 'val_macro_f1', 'val_macro_f1_std',
        'original_val_macro_f1', 'delta_vs_original',
        'model', 'alpha', 'fit_prior', 'norm',
        'ngram_range', 'min_df', 'use_safe_text_cleaning',
    ]
    top_cols = [c for c in top_cols if c in reval.columns]
    top = reval.sort_values('val_macro_f1', ascending=False).head(5)[top_cols]
    print('Top 5 by CV mean:')
    print(top.to_string(index=False))
    print('----------------------------')


def main():
    data = load_data()
    run_sweep(data)


if __name__ == '__main__':
    main()
